# Tabular foundation models vs. gradient boosting — a bake-off on the same leakage-audited split

This is a companion notebook to `credit_score_model.ipynb`, not a replacement for it. It answers one
question that notebook's own diagnostics raised but didn't settle: the ten models trained there
(LightGBM, XGBoost, CatBoost, Random Forest, a neural net, all tuned and untuned) landed within 0.006
macro-F1 of each other, and Optuna moved the best one by only +0.0007. That clustering was read as
"feature-limited, not algorithm-limited" — but that conclusion was only ever tested against more
gradient-boosted trees. This notebook tests it against a genuinely different architecture family:
**tabular foundation models** (in-context transformers, not boosted trees).

**Models compared, on the identical customer-grouped split as the main notebook:**
- `XGBoost` and `CatBoost` — the same gradient-boosting baseline, for reference
- `TabPFN-3.5` (Prior Labs) — in-context learning, no per-dataset training in the usual sense
- `TabICLv2` (via AutoGluon) — a newer in-context learner built for larger tabular datasets

**Read this before you get excited about a win:** this dataset is exactly the case where tabular
foundation models are reported to underperform — the holdout set is 2,500 customers **never seen
in training**, i.e. prediction for entities/IDs the model has no history for. Published TabPFN
benchmarks flag this "unseen entity" structure as the one case where tuned trees still tend to edge
out foundation models. So the expectation going in is calibrated: if XGBoost/CatBoost win here,
that's not a failed experiment, that's the expected result, honestly measured rather than assumed.
If a foundation model wins anyway, that's the more interesting finding.

**Before running:** add the "Credit Score Classification" dataset (same as the main notebook),
**Settings → Accelerator → GPU T4 ×2**, and **Settings → Internet → On** (the pip installs
below need it — easy to miss since the main notebook doesn't require internet).

Set `QUICK_TEST = True` in the config cell for a first run (~10–15 min, small subsample, 2 folds) to
confirm the whole notebook completes before spending the full budget. `QUICK_TEST = False` runs the
real comparison on the full data, 5 folds — expect well over an hour; TabPFN-3.5 and TabICLv2 are
both slower per row than a boosted tree.

In [79]:
!nvidia-smi

Thu Sep 17 06:26:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             34W /   70W |     163MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [80]:
# tabpfn: upgrading gets TabPFN-3.5 (open pip package, runs locally on this GPU -- not the paid API).
# autogluon.tabular[tabicl]: ships TabICLv2 behind AutoGluon's TabularPredictor.
# catboost: not preinstalled on the Kaggle image (matches the main notebook's own install cell).
!pip install -q --upgrade tabpfn
!pip install -q "autogluon.tabular[tabicl]"
!pip install -q catboost

In [81]:
import os, re, glob, gc, shutil, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, log_loss
from sklearn.utils.class_weight import compute_class_weight

import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CLASS_NAMES = ['Poor', 'Standard', 'Good']
TARGET_MAP  = {'Poor': 0, 'Standard': 1, 'Good': 2}

# ----------------------------------------------------------------------------------
# RUN BUDGET -- same philosophy as the main notebook's FAST_MODE: run cheap first.
#
# QUICK_TEST = True  : small group-safe subsample, 2 folds, short AutoGluon time limit.
#                      Confirms the whole notebook completes before the real run.
# QUICK_TEST = False : full training set, 5 folds -- what actually answers the question.
# ----------------------------------------------------------------------------------
QUICK_TEST = False

N_FOLDS               = 2 if QUICK_TEST else 5
SUBSAMPLE_N_CUSTOMERS = 1200 if QUICK_TEST else None   # None = use every training customer
AUTOGLUON_TIME_LIMIT  = 180 if QUICK_TEST else 900      # seconds, per fold, for TabICLv2

_T0 = time.time(); _LAST = [time.time()]
def section(name):
    now = time.time()
    print(f"\n{'=' * 72}")
    print(f"[{(now - _T0) / 60:6.1f} min total | +{(now - _LAST[0]) / 60:5.1f} min] {name}")
    print('=' * 72, flush=True)
    _LAST[0] = now

print(f"QUICK_TEST = {QUICK_TEST}  ->  {N_FOLDS} folds"
      + (f", subsampled to {SUBSAMPLE_N_CUSTOMERS} customers" if SUBSAMPLE_N_CUSTOMERS else ", full data"))
print("Set QUICK_TEST = False for the real comparison once this completes cleanly.")

QUICK_TEST = False  ->  5 folds, full data
Set QUICK_TEST = False for the real comparison once this completes cleanly.


In [82]:
def find_csv(name_pattern):
    matches = glob.glob(f"/kaggle/input/**/{name_pattern}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"No file matching '{name_pattern}' found under /kaggle/input. "
            f"Did you add the 'Credit Score Classification' dataset via + Add Input?"
        )
    return matches[0]

train_path = find_csv("train.csv")
print("Using train file:", train_path)

df_raw = pd.read_csv(train_path, low_memory=False)
print(df_raw.shape)
df_raw.head()

Using train file: /kaggle/input/datasets/parisrohan/credit-score-classification/train.csv
(100000, 28)


,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


## Same split as the main notebook

Group-aware, so no customer's rows land on both sides -- identical discipline to
`credit_score_model.ipynb`, same `SEED`, so these results sit on the same footing as that notebook's
0.7046 holdout macro-F1.

In [83]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, ho_idx = next(gss.split(df_raw, df_raw['Credit_Score'], groups=df_raw['Customer_ID']))

raw_train   = df_raw.iloc[tr_idx].reset_index(drop=True)
raw_holdout = df_raw.iloc[ho_idx].reset_index(drop=True)

assert set(raw_train['Customer_ID']).isdisjoint(set(raw_holdout['Customer_ID'])), "Customer leakage!"
print(f"Raw train:   {raw_train.shape}  ({raw_train['Customer_ID'].nunique():,} customers)")
print(f"Raw holdout: {raw_holdout.shape}  ({raw_holdout['Customer_ID'].nunique():,} customers)")
print("No customer overlap — confirmed.")

Raw train:   (80000, 28)  (10,000 customers)
Raw holdout: (20000, 28)  (2,500 customers)
No customer overlap — confirmed.


In [84]:
if SUBSAMPLE_N_CUSTOMERS is not None:
    _rng = np.random.default_rng(SEED)
    _cust = raw_train['Customer_ID'].unique()
    _keep = _rng.choice(_cust, size=min(SUBSAMPLE_N_CUSTOMERS, len(_cust)), replace=False)
    raw_train = raw_train[raw_train['Customer_ID'].isin(_keep)].reset_index(drop=True)
    print(f"QUICK_TEST subsample: {raw_train.shape} ({raw_train['Customer_ID'].nunique():,} customers)")

## Cleaning, imputation, feature engineering — reused verbatim

Same functions as the main notebook, copied in rather than re-derived, so this comparison isn't
accidentally running on a *different* feature set than the 0.7046 baseline was built on. See
`credit_score_model.ipynb` for the full reasoning behind each tier.

In [85]:
NUMERIC_COLS_TO_CLEAN = [
    'Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment',
    'Changed_Credit_Limit', 'Outstanding_Debt', 'Amount_invested_monthly',
    'Monthly_Balance', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate',
    'Num_Credit_Inquiries', 'Total_EMI_per_month', 'Monthly_Inhand_Salary',
    'Credit_Utilization_Ratio', 'Delay_from_due_date',
]

CATEGORICAL_COLS = ['Occupation', 'Credit_Mix', 'Payment_Behaviour', 'Payment_of_Min_Amount']

MONTH_ORDER = {m: i for i, m in enumerate(
    ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August',
     'September', 'October', 'November', 'December'], start=1)}


def clean_numeric(series):
    '''Strip junk characters from a string-typed numeric column and coerce to float.'''
    cleaned = (series.astype(str)
                     .str.replace(r'[^0-9.\-]', '', regex=True)
                     .replace('', np.nan)
                     .replace('-', np.nan))
    return pd.to_numeric(cleaned, errors='coerce')


def parse_credit_history_age(series):
    '''"22 Years and 1 Months" -> 265. Also handles years-only and months-only strings.'''
    def parse_one(x):
        if pd.isna(x):
            return np.nan
        s = str(x)
        y = re.search(r'(\d+)\s*Year', s)
        m = re.search(r'(\d+)\s*Month', s)
        if y is None and m is None:
            return np.nan
        return (int(y.group(1)) * 12 if y else 0) + (int(m.group(1)) if m else 0)
    return series.apply(parse_one)


def basic_clean(df):
    '''Stateless: type coercion, sentinel removal, month parsing. No fitted statistics.'''
    df = df.copy()

    for col in NUMERIC_COLS_TO_CLEAN:
        if col not in df.columns:
            continue
        if df[col].dtype == object:
            df[col] = clean_numeric(df[col])
        else:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df['Occupation']        = df['Occupation'].replace('_______', np.nan)
    df['Credit_Mix']        = df['Credit_Mix'].replace('_', np.nan)
    df['Payment_Behaviour'] = df['Payment_Behaviour'].replace('!@9#%8', np.nan)

    df['Credit_History_Age_Months'] = parse_credit_history_age(df['Credit_History_Age'])
    df['Month_Num'] = df['Month'].map(MONTH_ORDER)
    return df


def apply_outlier_rules(df, stats):
    '''Impossible / extreme values -> NaN. Quantile caps come from the fitted `stats`.'''
    df = df.copy()
    df.loc[(df['Age'] < 14) | (df['Age'] > 100), 'Age'] = np.nan
    df.loc[df['Annual_Income'] > stats['income_cap'], 'Annual_Income'] = np.nan
    df.loc[df['Total_EMI_per_month'] > stats['emi_cap'], 'Total_EMI_per_month'] = np.nan
    df.loc[(df['Num_Bank_Accounts'] < 0) | (df['Num_Bank_Accounts'] > 20), 'Num_Bank_Accounts'] = np.nan
    df.loc[df['Num_Credit_Card'] > 20, 'Num_Credit_Card'] = np.nan
    df.loc[df['Interest_Rate'] > 40, 'Interest_Rate'] = np.nan
    df.loc[(df['Num_of_Loan'] < 0) | (df['Num_of_Loan'] > 15), 'Num_of_Loan'] = np.nan
    df.loc[(df['Num_of_Delayed_Payment'] < 0) | (df['Num_of_Delayed_Payment'] > 30),
           'Num_of_Delayed_Payment'] = np.nan
    df.loc[df['Num_Credit_Inquiries'] > 50, 'Num_Credit_Inquiries'] = np.nan
    return df

In [86]:
# Portfolio regime (True): a customer's full history window is on file, so imputation and
# features may look across all of their months. Cold-start regime (False): only the past may
# be used. This one flag governs both the imputation below and the window features later.
USE_FULL_WINDOW = True

# Tier 1 -- attributes that should barely move month to month for the same person.
# Carrying the last/next observed value forward is the right call.
PANEL_COLS = ['Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary',
              'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate',
              'Num_of_Loan', 'Credit_Mix', 'Payment_Behaviour',
              'Payment_of_Min_Amount', 'Num_Credit_Inquiries']

# Tier 2 -- genuinely time-varying, so copying a neighbouring month is wrong, but the
# population median is far worse. Interpolate along the customer's own timeline instead.
PANEL_INTERP_COLS = ['Num_of_Delayed_Payment', 'Amount_invested_monthly',
                     'Changed_Credit_Limit', 'Monthly_Balance', 'Outstanding_Debt',
                     'Delay_from_due_date', 'Credit_Utilization_Ratio',
                     'Total_EMI_per_month']


def reconstruct_credit_history(df):
    '''`Credit_History_Age_Months - Month_Num` is constant within a customer -> exact recovery.'''
    df = df.copy()
    anchor = (df['Credit_History_Age_Months'] - df['Month_Num'])
    anchor = anchor.groupby(df['Customer_ID']).transform('median')
    df['Credit_History_Age_Months'] = df['Credit_History_Age_Months'].fillna(anchor + df['Month_Num'])
    df.loc[df['Credit_History_Age_Months'] < 0, 'Credit_History_Age_Months'] = np.nan
    return df


def panel_fill(df):
    '''Fill within each customer's own timeline first. Uses no population statistics.'''
    df = df.sort_values(['Customer_ID', 'Month_Num']).reset_index(drop=True)
    df = reconstruct_credit_history(df)

    # Tier 1: carry the customer's own observed value across their months.
    for col in PANEL_COLS:
        if col in df.columns:
            df[col] = df.groupby('Customer_ID')[col].transform(lambda s: s.ffill().bfill())

    # Tier 2: interpolate along the customer's timeline. Rows are already month-ordered, so
    # this is a temporally sensible estimate rather than a copied neighbour.
    # `interpolate` looks both ways, which is a portfolio-regime assumption -- under the strict
    # cold-start setting we may only carry the past forward.
    for col in PANEL_INTERP_COLS:
        if col in df.columns:
            g = df.groupby('Customer_ID')[col]
            if USE_FULL_WINDOW:
                df[col] = g.transform(
                    lambda s: s.interpolate(limit_direction='both')).astype(float)
            else:
                df[col] = g.transform(lambda s: s.ffill())
    return df


def global_fill(df, stats):
    '''Whatever a customer's own history could not supply, fall back to TRAIN statistics.'''
    df = df.copy()
    for col, val in stats['medians'].items():
        if col in df.columns:
            df[col] = df[col].fillna(val)
    for col, val in stats['modes'].items():
        if col in df.columns:
            df[col] = df[col].fillna(val)
    return df

In [87]:
LOAN_TYPES = ['Auto Loan', 'Credit-Builder Loan', 'Personal Loan', 'Home Equity Loan',
              'Mortgage Loan', 'Student Loan', 'Debt Consolidation Loan', 'Payday Loan']

ROLL_COLS = ['Delay_from_due_date', 'Num_of_Delayed_Payment', 'Outstanding_Debt',
             'Credit_Utilization_Ratio', 'Changed_Credit_Limit', 'Num_Credit_Inquiries',
             'Monthly_Balance']

# Full-window aggregates summarise a customer across ALL their months.
# Governed by USE_FULL_WINDOW, set alongside the imputation tiers above.
WINDOW_COLS = ['Delay_from_due_date', 'Num_of_Delayed_Payment', 'Outstanding_Debt',
               'Credit_Utilization_Ratio', 'Changed_Credit_Limit', 'Num_Credit_Inquiries',
               'Interest_Rate', 'Credit_Mix_ord', 'Monthly_Balance']


def loan_type_features(df):
    '''Multi-hot the comma-separated Type_of_Loan list, plus a distinct-type count.'''
    df = df.copy()
    s = df['Type_of_Loan'].fillna('').astype(str)
    for lt in LOAN_TYPES:
        col = 'Loan_' + lt.replace(' ', '_').replace('-', '_')
        df[col] = s.str.contains(re.escape(lt), case=False, regex=True).astype(np.int8)

    def count_types(x):
        x = x.strip()
        if x in ('', 'nan', 'Not Specified'):
            return 0
        parts = re.split(r',\s*and\s+|,\s*|\s+and\s+', x)
        return len([p for p in parts if p.strip() and p.strip() != 'Not Specified'])

    df['Num_Loan_Types'] = s.apply(count_types).astype(np.int16)
    return df


def build_features(df):
    '''All stateless or strictly within-customer. Safe to run per split side.'''
    df = df.sort_values(['Customer_ID', 'Month_Num']).reset_index(drop=True)

    # --- same-month ratios ---
    df['Debt_to_Income']       = df['Outstanding_Debt'] / (df['Annual_Income'] + 1)
    df['EMI_to_Income']        = (df['Total_EMI_per_month'] * 12) / (df['Annual_Income'] + 1)
    df['Investment_to_Income'] = (df['Amount_invested_monthly'] * 12) / (df['Annual_Income'] + 1)
    df['Loan_per_Account']     = df['Num_of_Loan'] / (df['Num_Bank_Accounts'] + 1)
    df['Debt_per_Loan']        = df['Outstanding_Debt'] / (df['Num_of_Loan'] + 1)
    df['Salary_Ratio']         = df['Monthly_Inhand_Salary'] * 12 / (df['Annual_Income'] + 1)
    df['Delay_per_Loan']       = df['Num_of_Delayed_Payment'] / (df['Num_of_Loan'] + 1)
    df['Credit_Age_per_Year']  = df['Credit_History_Age_Months'] / (df['Age'] * 12 + 1)

    df = loan_type_features(df)

    # numeric view of Credit_Mix, so it can be aggregated across a customer's window below
    df['Credit_Mix_ord'] = df['Credit_Mix'].map({'Bad': 0, 'Standard': 1, 'Good': 2}).fillna(1)

    # --- how much history does this row actually have behind it? ---
    df['months_of_history'] = df.groupby('Customer_ID').cumcount().astype(np.int16)

    # --- causal rolling stats: shift(1) BEFORE expanding, so no row sees its own present ---
    g = df.groupby('Customer_ID')
    for col in ROLL_COLS:
        hist_mean = g[col].transform(lambda s: s.shift(1).expanding().mean())
        hist_std  = g[col].transform(lambda s: s.shift(1).expanding().std())
        hist_max  = g[col].transform(lambda s: s.shift(1).expanding().max())
        # trajectory: computed BEFORE the NaN->0 fill, so month 1 stays NaN instead of
        # becoming a spurious "delta equal to the raw level"
        df[f'{col}_hist_mean']  = hist_mean
        df[f'{col}_hist_std']   = hist_std
        df[f'{col}_hist_max']   = hist_max
        df[f'{col}_delta_mean'] = df[col] - hist_mean
        df[f'{col}_diff']       = g[col].diff()

    hist_cols = [c for c in df.columns
                 if c.endswith(('_hist_mean', '_hist_std', '_hist_max', '_delta_mean', '_diff'))]
    df[hist_cols] = df[hist_cols].fillna(0)

    # --- full-window customer aggregates (PORTFOLIO REGIME ONLY) ---
    # These look across ALL of a customer's months, including later ones. That is legitimate
    # when you hold a complete history window on file, and invalid for a cold-start decision.
    # Still label-free -- no target information is involved, only features.
    if USE_FULL_WINDOW:
        for col in WINDOW_COLS:
            gc = df.groupby('Customer_ID')[col]
            df[f'{col}_cust_mean'] = gc.transform('mean')
            df[f'{col}_cust_std']  = gc.transform('std')
            df[f'{col}_cust_min']  = gc.transform('min')
            df[f'{col}_cust_max']  = gc.transform('max')
            df[f'{col}_cust_rng']  = df[f'{col}_cust_max'] - df[f'{col}_cust_min']
            df[f'{col}_vs_cust']   = df[col] - df[f'{col}_cust_mean']

    df = df.replace([np.inf, -np.inf], np.nan)
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)
    return df

## Encoding

One frame for everyone in this comparison. Foundation models and boosted trees alike get the same
one-hot matrix here -- CatBoost's native-categorical trick is skipped on purpose, so every model in
this bake-off sees literally identical inputs and the comparison isn't confounded by who got a better
feature representation.

In [88]:
DROP_COLS = ['ID', 'Name', 'SSN', 'Month', 'Credit_History_Age', 'Type_of_Loan',
             'Month_Num', 'Credit_Score', 'Customer_ID']

ORDINAL_MAPS = {
    'Credit_Mix':            ({'Bad': 0, 'Standard': 1, 'Good': 2}, 1),
    'Payment_of_Min_Amount': ({'No': 0, 'Yes': 1, 'NM': 0.5}, 0.5),
}
CAT_FEATURES = ['Occupation', 'Payment_Behaviour']


def encode(df_fe, ohe_columns=None, cat_columns=None):
    '''Returns (X_ohe, X_cat, y, groups). Pass the train column lists to align holdout.'''
    df = df_fe.copy()
    for col, (mapping, default) in ORDINAL_MAPS.items():
        df[col] = df[col].map(mapping).fillna(default).astype(float)

    y      = df['Credit_Score'].map(TARGET_MAP).astype(int)
    groups = df['Customer_ID'].copy()

    base = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

    # CatBoost frame: keep the categoricals as strings
    X_cat = base.copy()
    for c in CAT_FEATURES:
        X_cat[c] = X_cat[c].astype(str)
    if cat_columns is not None:
        X_cat = X_cat.reindex(columns=cat_columns)
        for c in CAT_FEATURES:
            X_cat[c] = X_cat[c].fillna('Unknown').astype(str)
        X_cat = X_cat.fillna(0)

    # One-hot frame for everything else
    X_ohe = pd.get_dummies(base, columns=CAT_FEATURES, drop_first=True)
    if ohe_columns is not None:
        X_ohe = X_ohe.reindex(columns=ohe_columns, fill_value=0)
    X_ohe = X_ohe.astype(np.float32)

    return X_ohe, X_cat, y, groups

In [89]:
def fit_transform_train(raw):
    stats = {}
    df = basic_clean(raw)
    stats['income_cap'] = df['Annual_Income'].quantile(0.999)
    stats['emi_cap']    = df['Total_EMI_per_month'].quantile(0.999)
    df = apply_outlier_rules(df, stats)
    df = panel_fill(df)
    num_cols = df.select_dtypes(include=[np.number]).columns
    stats['medians'] = df[num_cols].median().to_dict()
    stats['modes']   = {c: df[c].mode()[0] for c in CATEGORICAL_COLS if c in df.columns}
    df = global_fill(df, stats)
    return build_features(df), stats


def transform(raw, stats):
    df = basic_clean(raw)
    df = apply_outlier_rules(df, stats)
    df = panel_fill(df)
    df = global_fill(df, stats)
    return build_features(df)


fe_train, STATS = fit_transform_train(raw_train)
fe_holdout      = transform(raw_holdout, STATS)

# `_` is encode()'s CatBoost-native-categorical frame -- intentionally unused here (not a bug):
# every model in this comparison gets the identical one-hot matrix, so CatBoost doesn't get an
# unfair native-categorical advantage the others can't use. Don't "fix" this by wiring CatBoost
# back to X_cat -- that would silently break the apples-to-apples premise of this notebook.
X_train, _, y_train, groups_train = encode(fe_train)
OHE_COLUMNS = list(X_train.columns)
X_holdout, _, y_holdout, groups_holdout = encode(fe_holdout, OHE_COLUMNS)

assert set(groups_train).isdisjoint(set(groups_holdout)), "AUDIT FAIL: customer overlap"
assert list(X_train.columns) == list(X_holdout.columns), "AUDIT FAIL: column mismatch"

print(f"Train: {X_train.shape}   Holdout: {X_holdout.shape}")
print(f"NaNs -> train: {X_train.isna().sum().sum()}, holdout: {X_holdout.isna().sum().sum()}")

Train: (80000, 146)   Holdout: (20000, 146)
NaNs -> train: 0, holdout: 0


In [90]:
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
print("Class weights:", {CLASS_NAMES[k]: round(v, 3) for k, v in class_weight_dict.items()})

cv = list(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
          .split(X_train, y_train, groups=groups_train))

print("\nFold class balance (StratifiedGroupKFold keeps these even; plain GroupKFold does not):")
for i, (_, va) in enumerate(cv):
    print(f"  fold {i}: n={len(va):6,}  " +
          "  ".join(f"{CLASS_NAMES[c]}={ (y_train.iloc[va]==c).mean():.3f}" for c in classes))

Class weights: {'Poor': np.float64(1.146), 'Standard': np.float64(0.626), 'Good': np.float64(1.887)}

Fold class balance (StratifiedGroupKFold keeps these even; plain GroupKFold does not):
  fold 0: n=16,000  Poor=0.294  Standard=0.529  Good=0.177
  fold 1: n=15,992  Poor=0.285  Standard=0.546  Good=0.170
  fold 2: n=16,000  Poor=0.287  Standard=0.529  Good=0.184
  fold 3: n=16,008  Poor=0.295  Standard=0.532  Good=0.174
  fold 4: n=16,000  Poor=0.294  Standard=0.527  Good=0.179


In [91]:
def compute_metrics(y_true, proba):
    pred = proba.argmax(axis=1)
    return {
        'macro_f1':    f1_score(y_true, pred, average='macro'),
        'weighted_f1': f1_score(y_true, pred, average='weighted'),
        'accuracy':    accuracy_score(y_true, pred),
        'roc_auc_ovr': roc_auc_score(y_true, proba, multi_class='ovr', average='macro'),
        'log_loss':    log_loss(y_true, np.clip(proba, 1e-9, 1)),
    }


REG = {}   # name -> {'oof', 'holdout', 'fit_seconds', 'predict_seconds'}


def run_bakeoff(name, fit_fold, X, y, X_ho, cv, verbose=True):
    '''fit_fold(X_tr, y_tr, X_va, y_va, X_ho, fold_idx) -> (proba_va, proba_ho, fit_s, pred_s)
    Same OOF/holdout-averaging contract as the main notebook's run_cv, plus timing --
    this comparison is explicitly about training/inference cost, not just accuracy.'''
    oof = np.zeros((len(X), 3), dtype=np.float64)
    ho  = np.zeros((len(X_ho), 3), dtype=np.float64)
    fit_s = pred_s = 0.0

    for k, (i_tr, i_va) in enumerate(cv):
        p_va, p_ho, fs, ps = fit_fold(X.iloc[i_tr], y.iloc[i_tr], X.iloc[i_va], y.iloc[i_va], X_ho, k)
        oof[i_va] = p_va
        ho += p_ho / len(cv)
        fit_s += fs; pred_s += ps
        if verbose:
            print(f"  fold {k}: macro-F1 = {f1_score(y.iloc[i_va], p_va.argmax(1), average='macro'):.4f}"
                  f"  (fit {fs:.0f}s, predict {ps:.0f}s)")

    # Compute metrics BEFORE committing to REG -- if compute_metrics itself raises (e.g. a
    # brand-new library returning a malformed probability array), REG must not end up holding
    # a half-valid entry that then crashes the final comparison cell after the whole run.
    m_oof = compute_metrics(y, oof)
    m_ho  = compute_metrics(y_holdout, ho)
    REG[name] = {'oof': oof, 'holdout': ho, 'fit_seconds': fit_s, 'predict_seconds': pred_s}
    print(f"{name:14s} OOF macro-F1 = {m_oof['macro_f1']:.4f}  |  "
          f"holdout macro-F1 = {m_ho['macro_f1']:.4f}  |  "
          f"fit {fit_s:.0f}s total, predict {pred_s:.0f}s total")
    return m_oof, m_ho

## GPU probes

Same reasoning as the main notebook: Kaggle's pip-installed XGBoost/CatBoost GPU support varies by
image, and a foundation-model library failing to find a usable GPU mid-run is worse to discover late
than up front. Each library gets a tiny probe before the real run.

In [92]:
import xgboost as xgb
from catboost import CatBoostClassifier

_Xp, _yp = X_train.head(200), y_train.head(200)

def _probe(label, fn):
    try:
        fn(); print(f"  {label:10s} GPU: available"); return True
    except Exception as e:
        print(f"  {label:10s} GPU: unavailable -> CPU ({str(e)[:70]})"); return False

print("Probing GPU backends...")
XGB_GPU = _probe("XGBoost", lambda: xgb.XGBClassifier(
    n_estimators=5, tree_method='hist', device='cuda', verbosity=0).fit(_Xp, _yp))
CAT_GPU = _probe("CatBoost", lambda: CatBoostClassifier(
    iterations=5, task_type='GPU', devices='0', verbose=False).fit(_Xp, _yp))
TORCH_GPU = torch.cuda.is_available()
print(f"\nUsing -> XGBoost: {'GPU' if XGB_GPU else 'CPU'} | CatBoost: {'GPU' if CAT_GPU else 'CPU'} | "
      f"torch.cuda.is_available(): {TORCH_GPU} (TabPFN/TabICL use this)")

Probing GPU backends...
  XGBoost    GPU: available
  CatBoost   GPU: available

Using -> XGBoost: GPU | CatBoost: GPU | torch.cuda.is_available(): True (TabPFN/TabICL use this)


## Baselines — XGBoost, CatBoost

Simple, untuned configs. Not trying to reproduce the main notebook's Optuna-tuned numbers here --
the point of this notebook is architecture comparison, not squeezing the last 0.01 out of the trees.

In [93]:
cw = class_weight_dict   # reuse the one computed above -- don't recompute the same formula twice
xgb_w = [cw[c] for c in range(3)]

def fit_xgb(X_tr, y_tr, X_va, y_va, X_ho, k):
    t0 = time.time()
    m = xgb.XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=400,
                           learning_rate=0.05, max_depth=8, tree_method='hist',
                           device='cuda' if XGB_GPU else 'cpu', random_state=SEED, verbosity=0)
    m.fit(X_tr, y_tr, sample_weight=y_tr.map(cw).values)
    fs = time.time() - t0
    t0 = time.time()
    p_va, p_ho = m.predict_proba(X_va), m.predict_proba(X_ho)
    return p_va, p_ho, fs, time.time() - t0

def fit_cat(X_tr, y_tr, X_va, y_va, X_ho, k):
    t0 = time.time()
    m = CatBoostClassifier(iterations=400, learning_rate=0.05, depth=8, loss_function='MultiClass',
                            class_weights=xgb_w, random_seed=SEED, verbose=False,
                            task_type='GPU' if CAT_GPU else 'CPU', devices='0' if CAT_GPU else None)
    m.fit(X_tr, y_tr)
    fs = time.time() - t0
    t0 = time.time()
    p_va, p_ho = m.predict_proba(X_va), m.predict_proba(X_ho)
    return p_va, p_ho, fs, time.time() - t0

section("XGBoost")
run_bakeoff('XGBoost', fit_xgb, X_train, y_train, X_holdout, cv)
section("CatBoost")
run_bakeoff('CatBoost', fit_cat, X_train, y_train, X_holdout, cv)


[   2.0 min total | +  2.0 min] XGBoost
  fold 0: macro-F1 = 0.6842  (fit 10s, predict 0s)
  fold 1: macro-F1 = 0.6780  (fit 10s, predict 0s)
  fold 2: macro-F1 = 0.6873  (fit 11s, predict 0s)
  fold 3: macro-F1 = 0.6742  (fit 10s, predict 0s)
  fold 4: macro-F1 = 0.6854  (fit 10s, predict 0s)
XGBoost        OOF macro-F1 = 0.6818  |  holdout macro-F1 = 0.7033  |  fit 52s total, predict 1s total

[   2.9 min total | +  0.9 min] CatBoost
  fold 0: macro-F1 = 0.6831  (fit 5s, predict 0s)
  fold 1: macro-F1 = 0.6777  (fit 6s, predict 0s)
  fold 2: macro-F1 = 0.6843  (fit 6s, predict 0s)
  fold 3: macro-F1 = 0.6815  (fit 6s, predict 0s)
  fold 4: macro-F1 = 0.6924  (fit 6s, predict 0s)
CatBoost       OOF macro-F1 = 0.6838  |  holdout macro-F1 = 0.6929  |  fit 28s total, predict 0s total


({'macro_f1': 0.6837912446814318,
  'weighted_f1': 0.6933608116044511,
  'accuracy': 0.6901375,
  'roc_auc_ovr': np.float64(0.8640681393495336),
  'log_loss': 0.6829882509485726},
 {'macro_f1': 0.6928894254379004,
  'weighted_f1': 0.6985123706096164,
  'accuracy': 0.69675,
  'roc_auc_ovr': np.float64(0.8683883071990004),
  'log_loss': 0.6712050584008655})

## TabPFN-3.5

An in-context transformer, not a trained-per-dataset model in the usual sense -- `.fit()` mostly
caches the training rows for the forward pass, and the real cost is at predict time. Uses whatever
GPU `torch` finds; falls back to CPU (slow, but won't crash the notebook) if none is available.

Exact API surfaces on very new library releases can drift between when this was written and when you
run it -- this is wrapped defensively so a signature mismatch prints a clear error and the notebook
moves on to the next model rather than dying here. Check `tabpfn`'s current docs if this cell errors.

In [94]:
gc.collect(); torch.cuda.empty_cache() if TORCH_GPU else None  # release XGBoost/CatBoost GPU memory first

from tabpfn import TabPFNClassifier

# Conservative fallback cap in case TabPFN-3.5 still enforces a row-count ceiling internally --
# adjust against the currently-installed `tabpfn` version's docs; this only fires if .fit() raises.
TABPFN_MAX_ROWS_FALLBACK = 50_000

def fit_tabpfn(X_tr, y_tr, X_va, y_va, X_ho, k):
    t0 = time.time()
    device = 'cuda' if TORCH_GPU else 'cpu'
    try:
        m = TabPFNClassifier(device=device)
    except TypeError:
        m = TabPFNClassifier()   # older/newer signature without an explicit device kwarg
    try:
        m.fit(X_tr, y_tr)
    except Exception as e:
        # TabPFN's in-context-learning family has historically enforced a row-count ceiling;
        # if TabPFN-3.5 still does and this fold's training data exceeds it, retry once on a
        # random subsample rather than losing this model for the whole run. (Row-only subsample
        # is fine here -- this is purely an internal training-data cap for one fold's fit, not
        # the outer train/holdout split, so no group-safety requirement applies.)
        if len(X_tr) > TABPFN_MAX_ROWS_FALLBACK:
            print(f"    TabPFN-3.5 fit failed on {len(X_tr):,} rows ({type(e).__name__}: "
                  f"{str(e)[:120]}); retrying on a {TABPFN_MAX_ROWS_FALLBACK:,}-row subsample.")
            idx = np.random.default_rng(SEED).choice(len(X_tr), TABPFN_MAX_ROWS_FALLBACK, replace=False)
            m.fit(X_tr.iloc[idx], y_tr.iloc[idx])
        else:
            raise
    fs = time.time() - t0
    t0 = time.time()
    p_va, p_ho = m.predict_proba(X_va), m.predict_proba(X_ho)
    return p_va, p_ho, fs, time.time() - t0

section("TabPFN-3.5")
if not os.environ.get('TABPFN_TOKEN'):
    # Attaching a secret in Kaggle's Add-ons -> Secrets UI does NOT put it in os.environ --
    # it has to be pulled explicitly via kaggle_secrets. Try that before concluding it's missing.
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['TABPFN_TOKEN'] = UserSecretsClient().get_secret('TABPFN_TOKEN')
    except Exception:
        pass   # not running on Kaggle, or the secret isn't attached to this notebook
if not os.environ.get('TABPFN_TOKEN'):
    print("Skipping TabPFN-3.5: no TABPFN_TOKEN in the environment.")
    print("TabPFN >= 2.5 gates local weight downloads behind a one-time license acceptance:")
    print("  1. Open https://ux.priorlabs.ai and log in (or register)")
    print("  2. Accept the license under the 'Licenses' tab for the TabPFN-3.5 model")
    print("  3. Copy your token from the account page")
    print("  4. In Kaggle: Add-ons -> Secrets -> add TABPFN_TOKEN, then attach it to this notebook,")
    print("     or set it directly: os.environ['TABPFN_TOKEN'] = '<your token>' before this cell")
else:
    try:
        run_bakeoff('TabPFN-3.5', fit_tabpfn, X_train, y_train, X_holdout, cv)
    except Exception as e:
        print(f"TabPFN-3.5 failed: {type(e).__name__}: {str(e)[:300]}")
        print("Check the installed `tabpfn` version's current API (pip show tabpfn) and adjust fit_tabpfn.")


[   3.4 min total | +  0.5 min] TabPFN-3.5


tabpfn-v3.5-20260909.safetensors:   0%|          | 0.00/876M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

  fold 0: macro-F1 = 0.6854  (fit 11s, predict 1109s)
  fold 1: macro-F1 = 0.6828  (fit 5s, predict 1111s)
  fold 2: macro-F1 = 0.6835  (fit 5s, predict 1108s)
  fold 3: macro-F1 = 0.6697  (fit 5s, predict 1110s)
  fold 4: macro-F1 = 0.6867  (fit 5s, predict 1106s)
TabPFN-3.5     OOF macro-F1 = 0.6816  |  holdout macro-F1 = 0.6986  |  fit 31s total, predict 5544s total


## TabICLv2 (via AutoGluon)

AutoGluon's `TabularPredictor`, restricted to fitting *only* the `TABICL` model (`hyperparameters=
{'TABICL': {}}`) -- otherwise AutoGluon would train its own whole model zoo, which isn't what this
comparison is asking. `time_limit` bounds each fold so a slow fold can't blow the whole run open-ended,
same philosophy as the main notebook's Optuna `TUNE_TIMEOUT`.

In [97]:
gc.collect(); torch.cuda.empty_cache() if TORCH_GPU else None  # release TabPFN-3.5's GPU memory first

section("TabICLv2")
print("Skipping TabICLv2: AutoGluon's own memory estimator failed twice on this environment --")
print("64,000 rows estimated ~38.5 GB needed; 30,000 rows estimated ~68.1 GB (worse, not better,")
print("with fewer rows). That's a sign the ~28GB ceiling here is driven by the 146-column feature")
print("representation or the model's own architecture, not row count -- a third row-cap guess is")
print("unlikely to converge. Not evaluated in this bake-off; that's the honest, reportable result")
print("for this model at this feature-column count on Kaggle's standard-instance memory budget.")


[ 111.1 min total | + 14.8 min] TabICLv2
Skipping TabICLv2: AutoGluon's own memory estimator failed twice on this environment --
64,000 rows estimated ~38.5 GB needed; 30,000 rows estimated ~68.1 GB (worse, not better,
with fewer rows). That's a sign the ~28GB ceiling here is driven by the 146-column feature
representation or the model's own architecture, not row count -- a third row-cap guess is
unlikely to converge. Not evaluated in this bake-off; that's the honest, reportable result
for this model at this feature-column count on Kaggle's standard-instance memory budget.


## The comparison

Sorted by holdout macro-F1 — same headline metric as the main notebook, for direct comparability.
The main notebook's own published ensemble number is included as a static reference row (not
re-run here; see `credit_score_model_executed.ipynb`).

In [98]:
rows = []
for name, d in REG.items():
    try:
        m_oof = compute_metrics(y_train, d['oof'])
        m_ho  = compute_metrics(y_holdout, d['holdout'])
        rows.append({
            'model': name,
            'oof_macro_f1': m_oof['macro_f1'], 'holdout_macro_f1': m_ho['macro_f1'],
            'holdout_auc': m_ho['roc_auc_ovr'], 'holdout_acc': m_ho['accuracy'],
            'holdout_log_loss': m_ho['log_loss'],
            'fit_seconds': d['fit_seconds'], 'predict_seconds': d['predict_seconds'],
        })
    except Exception as e:
        # One model's stored predictions being malformed shouldn't blank the whole table --
        # report what's wrong and keep every other model's row.
        print(f"Skipping '{name}' in the comparison table: {type(e).__name__}: {str(e)[:200]}")

comparison = pd.DataFrame(rows).sort_values('holdout_macro_f1', ascending=False).set_index('model')
display(comparison.round(4))

print("\nReference (not re-run here) -- main notebook's published ensemble:")
print("  Blend + tuned trees   holdout_macro_f1=0.7046  holdout_auc=0.8715  (credit_score_model_executed.ipynb)")

if not QUICK_TEST:
    comparison.to_csv('/kaggle/working/tfm_bakeoff_results.csv')
    print("\nSaved: tfm_bakeoff_results.csv")

,oof_macro_f1,holdout_macro_f1,holdout_auc,holdout_acc,holdout_log_loss,fit_seconds,predict_seconds
model,,,,,,,
XGBoost,0.6818,0.7033,0.8678,0.7108,0.6511,51.9417,1.3187
TabPFN-3.5,0.6816,0.6986,0.8686,0.7122,0.6329,30.6510,5543.7270
CatBoost,0.6838,0.6929,0.8684,0.6968,0.6712,27.7049,0.4484



Reference (not re-run here) -- main notebook's published ensemble:
  Blend + tuned trees   holdout_macro_f1=0.7046  holdout_auc=0.8715  (credit_score_model_executed.ipynb)

Saved: tfm_bakeoff_results.csv


## Which metric actually decides this

**Primary: holdout macro-F1.** Kept as the same headline metric as the main notebook on purpose --
switching metrics between the two notebooks would make "did the foundation model win" an unanswerable
question. Macro-F1 stays imbalance-robust here for the same reason it was chosen originally (53/29/18
class split).

**Secondary: ROC-AUC (OvR).** Threshold-free ranking quality. Worth checking separately from macro-F1
because TabPFN/TabICL and the boosted trees are calibrated differently by construction -- AUC isolates
"does it rank risk correctly" from "did this specific decision threshold suit it," which matters when
comparing across architecture families rather than just hyperparameter settings within one.

**Not the deciding metric, but the one worth reporting anyway: `fit_seconds` / `predict_seconds`.**
This is the actual portfolio-relevant number if a foundation model doesn't clearly win on macro-F1 --
"comparable accuracy, but 5x the inference cost" is a real, useful finding, not a null result. Expect
the trees to be faster; if a foundation model is both slower *and* worse here, that's the honest
answer to "why not just use the shiny new one," backed by a number instead of an assumption.

**Log loss is reported but not trusted as a primary signal** -- the main notebook's own final model
already documents that class-weighting distorts calibration, so a log-loss comparison across
differently-weighted/calibrated model families needs the same caveat here.